# Donut fine-tune — InBody **270** v9 — Kaggle runner

Train only on the verified 5,000-sheet 270 corpus. PBF targets are serialized to one decimal place; runtime parsing remains numeric.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
BRANCH = 'feat/module-1-real-holdout-scorer'
REQUIRED_COMMIT = 'bb8a573'  # 270-only implementation plus exact v9 fingerprint inputs
from kaggle_secrets import UserSecretsClient
try:
    GH_TOKEN = UserSecretsClient().get_secret('GH_TOKEN')
    REPO = f'https://{GH_TOKEN}@github.com/QeekOw/InForm.git'
except Exception:
    REPO = 'https://github.com/QeekOw/InForm.git'
%cd /kaggle/working
!rm -rf /kaggle/working/repo
!git clone --branch $BRANCH --single-branch $REPO repo
%cd /kaggle/working/repo
!git checkout --force $REQUIRED_COMMIT
from pathlib import Path
_assets = Path('notebooks/kaggle_v9_fingerprint_inputs/src/inform/synthetic')
assert _assets.exists(), 'Missing fingerprint assets in Kaggle kernel package'
for _source in (_assets / '__init__.py', *_assets.joinpath('templates').glob('*.html')):
    _target = Path('src/inform/synthetic') / _source.relative_to(_assets)
    _target.parent.mkdir(parents=True, exist_ok=True)
    _target.write_bytes(_source.read_bytes())
!pip install -q -e '.[training]'


In [ ]:
# Attach the v9 training dataset as Kaggle input. Validate its manifests,
# fingerprints, pairs, and exact count before spending GPU time.
import glob, json, os, shutil, subprocess, sys

manifests = glob.glob('/kaggle/input/**/dataset.*.json', recursive=True)
if not manifests:
    archives = glob.glob('/kaggle/input/**/*.tar', recursive=True)
    assert len(archives) == 1, f'Expected one v9 training archive; found {len(archives)}'
    shutil.unpack_archive(archives[0], '/kaggle/tmp/v9-train')
    manifests = glob.glob('/kaggle/tmp/v9-train/**/dataset.*.json', recursive=True)
data_dirs = sorted({os.path.dirname(manifest) for manifest in manifests})
assert len(data_dirs) == 1, f'Expected one verified dataset root; found {len(data_dirs)}'
DATA_DIR = data_dirs[0]
for manifest in manifests:
    with open(manifest, encoding='utf-8') as handle:
        assert json.load(handle)['devices'] == ['inbody_270'], f'Non-270 manifest: {manifest}'
subprocess.run([sys.executable, 'scripts/verify_dataset.py', DATA_DIR, '--expect', '5000'], check=True)
sheets = sorted(glob.glob(f'{DATA_DIR}/*.jpg'))
assert len(sheets) == 5000, f'Expected 5,000 training sheets; found {len(sheets)}'
assert all('inbody_270' in os.path.basename(sheet) for sheet in sheets), 'v9 accepts only InBody 270 sheets'
print('DATA_DIR =', DATA_DIR, '| InBody 270 sheets:', len(sheets))


In [ ]:
# Fresh from donut-base. Batch 1 + gradient accumulation 4 fits the 270 canvas
# on a 16 GB T4; three epochs fit within Kaggle's 12-hour session limit.
CHECKPOINT_DIR = '/kaggle/working/donut-270-v9'
!python -m inform.training.train \
  --data-dir "$DATA_DIR" --output-dir $CHECKPOINT_DIR \
  --model-name-or-path naver-clova-ix/donut-base \
  --epochs 3 --batch-size 1 --gradient-accumulation-steps 4 \
  --dataloader-num-workers 4 --learning-rate 3e-5


In [ ]:
# Resume only after a session cap. Attach the previous output as input, then
# run this cell instead of the fresh-training cell.
# import glob, shutil
# previous = glob.glob('/kaggle/input/**/donut-270-v9', recursive=True)[0]
# shutil.copytree(previous, CHECKPOINT_DIR, dirs_exist_ok=True)
# !python -m inform.training.train --data-dir "$DATA_DIR" --output-dir $CHECKPOINT_DIR --model-name-or-path naver-clova-ix/donut-base --epochs 3 --batch-size 1 --gradient-accumulation-steps 4 --dataloader-num-workers 4 --learning-rate 3e-5 --resume


In [ ]:
!du -sh /kaggle/working/donut-270-v9/* | sort -h
!ls -la /kaggle/working/donut-270-v9


## After the run

Score every epoch checkpoint on both development sets: the original fixed 12 sheets and the five-sheet v8 failure set. A checkpoint can become the frozen candidate only if the original set keeps PBF at least 10/12, both arm values at 12/12, and flagged/unread no worse than v5; the five-sheet set must have zero refusals and beat or match v5's 28/35 core and 23/30 critical fields. Only then score that frozen candidate once against a fresh independent confirmation set of at least five new InBody 270 sheets. v5 remains the default until it passes.